In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Jahangirpuri_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,361.0,246.0,280.0,158.0,258.0,278.0,202.0,66.0,140.0,195.0,369.0,316.0
1,2,362.0,284.0,149.0,200.0,274.0,NaN,225.0,121.0,106.0,190.0,332.0,318.0
2,3,365.0,279.0,155.0,213.0,343.0,NaN,210.0,59.0,120.0,185.0,414.0,304.0
3,4,423.0,351.0,160.0,188.0,336.0,249.0,104.0,81.0,85.0,212.0,422.0,205.0
4,5,362.0,274.0,154.0,234.0,369.0,312.0,60.0,59.0,93.0,178.0,413.0,210.0
5,6,351.0,223.0,154.0,221.0,321.0,213.0,94.0,81.0,125.0,170.0,409.0,251.0
6,7,387.0,253.0,256.0,310.0,351.0,251.0,62.0,160.0,86.0,172.0,434.0,278.0
7,8,384.0,219.0,188.0,251.0,269.0,266.0,63.0,53.0,128.0,221.0,418.0,349.0
8,9,367.0,194.0,178.0,217.0,187.0,258.0,NaN,52.0,158.0,212.0,381.0,225.0
9,10,318.0,362.0,212.0,240.0,NaN,223.0,NaN,98.0,107.0,178.0,377.0,287.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,361.0,246.000000,280.0,158.000000,258.000000,278.000000,115.096774,66.000000,140.0,195.0,369.000000,316.0
1,2,362.0,284.000000,149.0,200.000000,274.000000,211.310345,115.096774,121.000000,106.0,190.0,332.000000,318.0
2,3,365.0,279.000000,155.0,213.000000,343.000000,211.310345,115.096774,59.000000,120.0,185.0,414.000000,304.0
3,4,423.0,351.000000,160.0,188.000000,336.000000,249.000000,104.000000,81.000000,85.0,212.0,422.000000,205.0
4,5,362.0,274.000000,154.0,234.000000,369.000000,211.310345,60.000000,59.000000,93.0,178.0,413.000000,210.0
5,6,351.0,223.000000,154.0,221.000000,321.000000,213.000000,94.000000,81.000000,125.0,170.0,409.000000,251.0
6,7,387.0,253.000000,256.0,310.000000,351.000000,251.000000,62.000000,92.147059,86.0,172.0,434.000000,278.0
7,8,384.0,219.000000,188.0,251.000000,269.000000,266.000000,63.000000,53.000000,128.0,221.0,418.000000,349.0
8,9,367.0,194.000000,178.0,217.000000,187.000000,258.000000,115.096774,52.000000,158.0,212.0,381.000000,225.0
9,10,318.0,362.000000,212.0,240.000000,233.228571,223.000000,115.096774,98.000000,107.0,178.0,377.000000,287.0
